# Transformers and Tap Changers

----

The 61970 Wires package defines several classes for representing the different components of transformers. The first is PowerTransformer, which represents the electrical network representation of a transformer for both balanced and unbalanced circuits. The second is TransformerTank, which refers to the assembly of two or more windings placed inside a tank and can be used to model single-phase and three-phase transformers. TransformerEnd is the conducting connection point of a transformer and corresponds to the terminal of a particular winding.  As with all ConductingEquipment, the PowerTransformer is connected through a set of Terminal objects, with the particular winding indicated by the TransformerEnd.endNumber attribute. The highest voltage winding should have an endNumber of 1. The endNumber does not need to match the ACDCTerminal.sequenceNumber attribute of the Terminal to which the transformer is connected. 

The BaseVoltage and Terminal are associated with the TransformerEnd of each winding, rather than with the PowerTransformer itself. Figure below shows the associations between the different classes used to specify transformer parameters and windings.

PowerTransformer objects may be modeled with or without specifying TransformerTank objects. In both cases the PowerTransformer.vectorGroup attribute for protective relaying should be specified according to IEC transformer standards (e.g., Dy1 for many substation transformers). 


The case without specifying TransformerTank objects is most suitable for balanced three-phase transformers that will not reference any reusable asset catalog data. This approach is typically used for transmission system modeling, where each transformer is unique. Each winding will have a PowerTransformerEnd that associates to both a Terminal and a BaseVoltage, and the parent PowerTransformer. The impedance and admittance parameters are defined by reverse-associated TransformerMeshImpedance between each pair of windings, and a reverse-associated TransformerCoreAdmittance for one winding. The units for these are ohms and siemens based on the winding voltage, rather than per-unit. WindingConnection is similar to PhaseShuntConnectionKind, adding Z and Zn for zig-zag connections and A for autotranformers. TransformerStarImpedance is used for conversion of three-winding transformers to separate two-winding equivalents, which is common practice in numerous power flow solvers.

If the transformer is unbalanced in any way, then TransformerTankEnd is used instead of PowerTransformerEnd, and then one or more TransformerTank objects may be used in the parent PowerTransformer. Some of the use cases are 1) center-tapped secondary, 2) open-delta and 3) EHV transformer banks. Tank-level modeling is also required if using catalog data to specify physical equipment ratings, etc. through the AssetInfo package. 



In [1]:
import os
from cimgraph import utils
from mermaid import Mermaid
import cimgraph.data_profile.cimhub_2023 as cim
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'

In [2]:
diagram_text = utils.get_mermaid([cim.ConductingEquipment, cim.PowerTransformer, cim.TransformerTank,
                                  cim.TransformerEnd, cim.PowerTransformerEnd, cim.TransformerTankEnd,
                                  cim.TransformerMeshImpedance, cim.TransformerCoreAdmittance, cim.TransformerStarImpedance,
                                  cim.Terminal, cim.BaseVoltage, cim.WindingConnection, cim.PhaseCode])
Mermaid(diagram_text)

Many distribution software packages use the concept of catalog data, aka library data, especially for lines and transformers. This concept is implemented in CIM through the ability to define a single set of class definitions using the IEC 61968 AssetInfo package to save a large amount of space when defining customer secondary transformers (which typically comprise hundreds or thousands of identical poletop and pad-mounted installations). A particular transformer design and rating is then defined by creating PowerTransformerInfo and TransformerTankInfo library objects that are associated with each type of specification objects. 

The rated voltage, rated current, and resistance of each winding are defined as attributes of a TransformerEndInfo object that is created for each transformer winding. It is important that the TransformerEndInfo.endNumber of the physical asset match the TransformerEnd.endNumber of its representation in the electrical circuit. The shunt admittances are defined by NoLoadTest on a winding / end, with usually just one such test. The impedances are defined by a set of attributes of ShortCircuitTest; one winding / end will be energized, and one or more of the others will be grounded in these tests. The complete list of asset properties is summarized in Figure below. Note that these classes are associated with TransformerTankInfo (not PowerTransformerInfo) because transformer testing is done on tanks.

In [3]:
diagram_text = utils.get_mermaid([cim.PowerTransformerInfo, cim.TransformerTankInfo, cim.TransformerEndInfo,
                                  cim.TransformerTest, cim.OpenCircuitTest, cim.ShortCircuitTest, cim.NoLoadTest,
                                  cim.WindingConnection])
Mermaid(diagram_text)

The TapChanger class is used to model both phase-shifting transformers and voltage regulators through the PhaseTapChanger and RatioTapChanger classes, which are associated with the particular TransformerEnd. The highest, lowest, and neutral tap positions available are specified as positive integers such that a TapChanger with 16 tap positions would have attributes of lowStep set to 0, neutralStep set to 8, and highStep set to 16. If a particular application uses a range of -8 to 8 for the same transformer tap range, it is the responsibility of application to convert the tap ranges to the format used internally. 

If a voltage regulator uses line drop compensation, then those parameters will be defined as attributes of the TapChangerControl class, which inherits from RegulatingControl. RegulatingControl is a higher-level class that is used to specify the control mode and setpoints for numerous types of RegulatingCondEq, such capacitors, reactors, SVC, and generator automatic voltage regulation controls. Whether a particular device is regulating voltage, activePower, reactivePower, etc. is specified by the RegulatingControl:mode attribute. Other control settings, such as targetValue, targetDeadband, etc.  are also attributes of RegulatingControl, as shown in Figure below.

In summary, a single-phase line voltage regulator modeled in CIM includes a PowerTransformer, a TransformerTank, a TransformerTankEnd, a RatioTapChanger, and a TapChangerControl. The CT and PT parameters of a voltage regulator can only be described via the AssetInfo mechanism, described below. The RegulationControl.mode must be voltage. Older CIM versions used the tculControlMode attribute, which is now deprecated. 

The AssetInfo package in the 61968 package defines the TapChangerInfo class with a set of attributes ctRating, ctRatio, and ptRatio needed for line drop compensator settings in voltage regulators. Catalog data is a one-to-many relationship. In this case, many TapChangers can share the same TapChangerInfo data, which saves space and provides consistency. Older versions of CIM had many-to-many catalog relationships, but now only one AssetDataSheet may be associated per Equipment.


In [4]:
diagram_text = utils.get_mermaid([cim.TransformerEnd, cim.PowerTransformerEnd, cim.TransformerTankEnd,
                                  cim.TapChanger, cim.RatioTapChanger, cim.TapChangerControl, cim.RegulatingControl,
                                  cim.RegulatingControlModeKind, cim.PhaseCode])
Mermaid(diagram_text)

Some examples are discussed below.

In [5]:
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
import cimgraph.data_profile.cimhub_2023 as cim
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'

In [6]:
network = FeederModel(connection=XMLFile(filename='../sample_models/ieee13.xml'), container=None)

Example 1: How many triplex secondary transformers are in the model?

In [7]:
# A split-phase secondary transformer has one TransformerTank, three Terminals,
# and no PowerTransformerEnd objects (those are used for three-phase windings)
results = set()

for power_transformer in network.list_by_class(cim.PowerTransformer):
    if (len(power_transformer.TransformerTanks) == 1
            and len(power_transformer.Terminals) == 3
            and len(power_transformer.PowerTransformerEnd) == 0):
        results.add(power_transformer.name)

print(results)

{'tpoletop'}


Example 2: Which split-phase transformers are connected to phase B? Do not include voltage regulators.

In [8]:
# Split-phase transformers have one tank and three terminals; the high-side
# winding connects to phase A, B, or C and the low-side to phases s1/s2
results = []

for power_transformer in network.list_by_class(cim.PowerTransformer):
    if len(power_transformer.TransformerTanks) == 1 and len(power_transformer.Terminals) == 3:
        transformer_tank = power_transformer.TransformerTanks[0]
        for tank_end in transformer_tank.TransformerTankEnds:
            if 'B' in str(tank_end.orderedPhases):
                results.append(power_transformer.name)

print(results)

['tpoletop']


Example 3: What are the voltage ratings of each winding for the single-phase transformer tank with mRID "17A934C7-1510-481F-BAD7-189058957FF1?

In [9]:
transformer_tank = network.get_object(mRID='17A934C7-1510-481F-BAD7-189058957FF1')
results = []

# Winding ratings come from the TransformerEndInfo datasheets on the tank's TransformerTankInfo
if transformer_tank.TransformerTankInfo is not None:
    for end_info in transformer_tank.TransformerTankInfo.TransformerEndInfos:
        # endNumber 1 is the high-side winding; 2 and 3 are the low-side ends
        results.append({'end_number': end_info.endNumber,
                        'rated_voltage': end_info.ratedU})

print(results)

[{'end_number': 1, 'rated_voltage': 2400.0}, {'end_number': 2, 'rated_voltage': 120.0}, {'end_number': 3, 'rated_voltage': 120.0}]


Example 4: How many three-winding transformers are in the model?

In [10]:
# A three-winding transformer has three Terminals and three PowerTransformerEnd objects
results = []

for power_transformer in network.list_by_class(cim.PowerTransformer):
    if len(power_transformer.Terminals) == 3 and len(power_transformer.PowerTransformerEnd) == 3:
        results.append(power_transformer.name)

print(results)

['sub3']


Example 5: How many two-winding transformers are in the model?

In [11]:
# A two-winding transformer has two Terminals and two PowerTransformerEnd objects
results = []

for power_transformer in network.list_by_class(cim.PowerTransformer):
    if len(power_transformer.Terminals) == 2 and len(power_transformer.PowerTransformerEnd) == 2:
        results.append(power_transformer.name)

print(results)

['xfm1']


Example 6: What are the names of buses that transformer name 'xfm1' is connected to?

In [12]:
name = 'xfm1'
results = []

# graph path: PowerTransformer -> Terminals -> ConnectivityNode
power_transformer = network.find_by_attribute(cim.PowerTransformer, 'name', name)[0]
for terminal in power_transformer.Terminals:
    results.append(terminal.ConnectivityNode.name)

print(results)

['xf1', '634']


Example 7: What is the nominal voltage ratings of three-phase transformer with mRID "1E6B5C97-C4E8-4CED-B9A5-6E69F389DA93?

In [13]:
power_transformer = network.get_object(mRID='1E6B5C97-C4E8-4CED-B9A5-6E69F389DA93')
results = []

# Each winding's nominal voltage is the PowerTransformerEnd.ratedU attribute
for power_transformer_end in power_transformer.PowerTransformerEnd:
    results.append(power_transformer_end.ratedU)

print(results)

[4160.0, 480.0]


Example 8: What is the apparent power rating of each three-phase transformer in the feeder?

In [14]:
results = []

# Three-phase transformers are the ones with PowerTransformerEnd windings
for power_transformer in network.list_by_class(cim.PowerTransformer):
    if power_transformer.PowerTransformerEnd:
        results.append({'xfmr_name': power_transformer.name,
                        'VA_rating': [end.ratedS for end in power_transformer.PowerTransformerEnd]})

print(results)

[{'xfmr_name': 'xfm1', 'VA_rating': [500000.0, 500000.0]}, {'xfmr_name': 'sub3', 'VA_rating': [5000000.0, 5000000.0, 1000000.0]}]


Example 9: How is the three-phase transformer named 'xfm1' connected?

In [15]:
name = 'xfm1'
results = {}

power_transformer = network.find_by_attribute(cim.PowerTransformer, 'name', name)[0]
# vectorGroup describes the overall winding connection (e.g. Dy1)
results['vector_group'] = str(power_transformer.vectorGroup)

for power_transformer_end in power_transformer.PowerTransformerEnd:
    # endNumber 1 is the high-side winding, 2 is the low-side; each has a connectionKind
    if power_transformer_end.endNumber == 1:
        results['high_side_connection'] = str(power_transformer_end.connectionKind)
    elif power_transformer_end.endNumber == 2:
        results['low_side_connection'] = str(power_transformer_end.connectionKind)

print(results)

{'vector_group': 'Yy', 'high_side_connection': 'WindingConnection.Y', 'low_side_connection': 'WindingConnection.Y'}


Example 10: How many windings does transformer with mRID "1E6B5C97-C4E8-4CED-B9A5-6E69F389DA93" have?

In [16]:
power_transformer = network.get_object(mRID='1E6B5C97-C4E8-4CED-B9A5-6E69F389DA93')

# Each winding is one PowerTransformerEnd
print(len(power_transformer.PowerTransformerEnd))

2


Example 11: What is the impedance of the transformer with mRID "1E6B5C97-C4E8-4CED-B9A5-6E69F389DA93"? Express the impedance on the basis of the nodes connected to each winding.

In [17]:
power_transformer = network.get_object(mRID='1E6B5C97-C4E8-4CED-B9A5-6E69F389DA93')
results = {}

# Walk each winding; impedance between windings is TransformerMeshImpedance,
# and the magnetizing branch is TransformerCoreAdmittance
for power_transformer_end in power_transformer.PowerTransformerEnd:
    if power_transformer_end.endNumber == 1:
        results['high_side_bus'] = power_transformer_end.Terminal.ConnectivityNode.name
        results['high_side_resistance'] = power_transformer_end.r

        # The high-side holds the FromMeshImpedance to the low-side winding
        if power_transformer_end.FromMeshImpedance:
            mesh_impedance = power_transformer_end.FromMeshImpedance[0]
            results['x'] = mesh_impedance.x
            results['r'] = mesh_impedance.r

        if power_transformer_end.CoreAdmittance is not None:
            results['b'] = power_transformer_end.CoreAdmittance.b
            results['g'] = power_transformer_end.CoreAdmittance.g

    elif power_transformer_end.endNumber == 2:
        results['low_side_bus'] = power_transformer_end.Terminal.ConnectivityNode.name
        results['low_side_resistance'] = power_transformer_end.r

print(results)

{'high_side_bus': 'xf1', 'high_side_resistance': 0.1903616, 'x': 0.692224, 'r': 0.3807232, 'b': 0.0, 'g': 0.0, 'low_side_bus': '634', 'low_side_resistance': 0.0025344}
